In [12]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np

In [13]:
#=========
# Setup
#=========
df = pd.DataFrame({
    "customer_id": [1 , 2 , 3 , 4 , 5 , 6 , 7 , 8] ,
    "region": ["East" , "West" , "West" , "North" , "East" , None , "Central" , "East"] ,
    "channel": ["Online" , "Retail" , "Online" , "Online" , "Retail" , "Retail" , "Online" , "Retail"] ,
    "sales": [120 , 80 , 5000 , 110 , 150 , 90 , 60 , 200] ,
    "returns": [5 , 2 , 60 , 6 , 4 , 1 , 7 , 0] ,
})
df["return_rate"] = df["returns"] / df["sales"]
df

,customer_id,region,channel,sales,returns,return_rate
0,1,East,Online,120,5,0.041667
1,2,West,Retail,80,2,0.025000
2,3,West,Online,5000,60,0.012000
3,4,North,Online,110,6,0.054545
4,5,East,Retail,150,4,0.026667
5,6,None,Retail,90,1,0.011111
6,7,Central,Online,60,7,0.116667
7,8,East,Retail,200,0,0.000000


In [14]:
#=========================================
# Case 1) Encoding (get_dummies)
#=========================================
case1 = pd.get_dummies(df[["region"]] , prefix = "region")
case1

,region_Central,region_East,region_North,region_West
0,False,True,False,False
1,False,False,False,True
2,False,False,False,True
3,False,False,True,False
4,False,True,False,False
5,False,False,False,False
6,True,False,False,False
7,False,True,False,False


In [15]:
#======================================
# Case 2) Production-friendly dummies
#======================================
case2 = pd.get_dummies(df[["region" , "channel"]] , prefix = ["region" , "ch"] , dummy_na = True , drop_first = True)
case2

,region_East,region_North,region_West,region_nan,ch_Retail,ch_nan
0,True,False,False,False,False,False
1,False,False,True,False,True,False
2,False,False,True,False,False,False
3,False,True,False,False,False,False
4,True,False,False,False,True,False
5,False,False,False,True,True,False
6,False,False,False,False,False,False
7,True,False,False,False,True,False


In [16]:
#=================================
# Case 3) Fixed bins with cut()
#=================================
bins = [0 , 100 , 200 , 1000 , np.inf]
labels = ["0-100" , "100-200" , "200-1000" , "1000+"]
df["sales_band"] = pd.cut(df["sales"] , bins = bins , labels = labels , include_lowest = True)
df[["sales" , "sales_band"]]

,sales,sales_band
0,120,100-200
1,80,0-100
2,5000,1000+
3,110,100-200
4,150,100-200
5,90,0-100
6,60,0-100
7,200,100-200


In [17]:
#======================================
# Case 4) Quantile bins with qcut()
#======================================
df["sales_quantile"] = pd.qcut(df["sales"] , q = 4 , labels = ["Q1" , "Q2" , "Q3" , "Q4"] , duplicates = "drop")
df[["sales" , "sales_quantile"]]

,sales,sales_quantile
0,120,Q3
1,80,Q1
2,5000,Q4
3,110,Q2
4,150,Q3
5,90,Q2
6,60,Q1
7,200,Q4


In [18]:
#====================================
# Case 5) Cap outliers with clip()
#====================================
cap = df["sales"].quantile(0.95)
df["sales_capped"] = df["sales"].clip(upper = cap)
df[["sales" , "sales_capped"]]

/var/folders/96/5mw18qzx70v4myyr662bvplr0000gn/T/ipykernel_85513/470085321.py:5: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["sales_capped"] = df["sales"].clip(upper = cap)


,sales,sales_capped
0,120,120
1,80,80
2,5000,3320
3,110,110
4,150,150
5,90,90
6,60,60
7,200,200


In [19]:
#===========================================
# Case 6) Conditional features with mask()
#===========================================
df["sales_if_low_returns"] = df["sales"].mask(df["return_rate"] > 0.10 , np.nan)
df["return_rate_clean"] = df["return_rate"].mask(df["return_rate"] > 0.50 , 0)
df[["sales" , "returns" , "return_rate" , "sales_if_low_returns" , "return_rate_clean"]].round(2)

features = pd.concat(
    [
        df[["customer_id" , "sales" , "sales_capped" , "return_rate_clean" , "sales_band" , "sales_quantile"]] ,
        pd.get_dummies(df[["region" , "channel"]] , prefix = ["region" , "ch"] , dummy_na = True) ,
    ] ,
    axis = 1 ,
)
features.head()

,sales,returns,return_rate,sales_if_low_returns,return_rate_clean
0,120,5,0.04,120.0,0.04
1,80,2,0.02,80.0,0.02
2,5000,60,0.01,5000.0,0.01
3,110,6,0.05,110.0,0.05
4,150,4,0.03,150.0,0.03
5,90,1,0.01,90.0,0.01
6,60,7,0.12,NaN,0.12
7,200,0,0.00,200.0,0.00


,customer_id,sales,sales_capped,return_rate_clean,sales_band,sales_quantile,region_Central,region_East,region_North,region_West,region_nan,ch_Online,ch_Retail,ch_nan
0,1,120,120,0.041667,100-200,Q3,False,True,False,False,False,True,False,False
1,2,80,80,0.025000,0-100,Q1,False,False,False,True,False,False,True,False
2,3,5000,3320,0.012000,1000+,Q4,False,False,False,True,False,True,False,False
3,4,110,110,0.054545,100-200,Q2,False,False,True,False,False,True,False,False
4,5,150,150,0.026667,100-200,Q3,False,True,False,False,False,False,True,False
